# 02 · Feature engineering as a research gate

**Decision:** The declared feature-research phase is complete. Retain the original frozen centroid for unseen policies. The 323-fit four-policy campaign, targeted semantic comparisons and fixed complementarity control support diminishing returns within this scope. Independent confirmation and product delivery remain separate gates.

This notebook explains candidate generation, training-only screening, matched ablations, negative findings and the stopping decision. The retained centroid reaches 0.7042 transfer AUC versus 0.4728 for the matched lexical reference. Compact semantic comparisons give the strongest family addition. A higher feature count is not the selection criterion.

In [ ]:
import os
from pathlib import Path
import pandas as pd
from IPython.display import display
from jigsaw_rules.review import public_evidence
from jigsaw_rules.runtime import environment

root = Path(os.environ.get("JIGSAW_ROOT", Path.cwd())).resolve()
if root.name == "notebooks":
    root = root.parent
baseline = public_evidence(root, "baseline")
semantic = public_evidence(root, "semantic")
assert baseline["training_sha256"] == semantic["training_sha256"]
print("Historical references: 2,029 rows, two policies; expanded research is a separate cohort.")
print("Aggregate checksums verified. This notebook performs no model fitting.")
protocols = {"seen_rule": "Familiar rules", "heldout_rule": "Held-out rule"}

def metric_table(records, heldout=False):
    return pd.DataFrame([{"Representation": r["model"], "Validation": protocols[r["protocol"]],
        "Rule macro AUC": r["metrics"]["rule_macro_auc"],
        "Log loss": r["metrics"]["log_loss"], "Brier": r["metrics"]["brier"],
        "Average precision": r["metrics"]["average_precision"]}
        for r in records if not heldout or r["protocol"] == "heldout_rule"]).round(4)


In [ ]:
import sys
sys.path.insert(0, str(root / "scripts"))
from build_research_report import display_figure
from build_release_report import display_boundary
from jigsaw_rules.features import feature_evidence
from jigsaw_rules.research import research_evidence
from jigsaw_rules.diagnostics import diagnostic_evidence
from jigsaw_rules.pairs import pairs_evidence
from jigsaw_rules.robustness import robustness_evidence
from jigsaw_rules.gate import feature_gate
from jigsaw_rules.instructions import instruction_evidence
from jigsaw_rules.released import released_evidence
from jigsaw_rules.expanded import expanded_evidence
from jigsaw_rules.retrieval import retrieval_evidence
from jigsaw_rules.resolution import resolution_evidence
from jigsaw_rules.formatting import formatting_evidence
from jigsaw_rules.feature_decision import decision_evidence
from build_expanded_report import display_figure as display_expanded
from build_formatting_report import display_figure as display_formatting

expanded = expanded_evidence(root)
retrieval = retrieval_evidence(root)
resolution = resolution_evidence(root)
formatting = formatting_evidence(root)
decision = decision_evidence(root)
assert expanded is not None and retrieval is not None and resolution is not None
assert formatting is not None and decision is not None
controls = feature_evidence(root)
research = research_evidence(root)
sensitivity = diagnostic_evidence(root)
pairs = pairs_evidence(root)
robustness = robustness_evidence(root)
assert all(item is not None for item in (controls, research, sensitivity, pairs, robustness))
gate = feature_gate(root)
instructions = instruction_evidence(root)
released = released_evidence(root)
assert released is not None
print("Verified expanded study:", expanded["metadata"]["run_id"])


## 1 · The research question and protected boundary
The historical search covered two policies. The host's released corpus now permits 11,135 development rows across advertising, legal advice, medical advice and illegal-activity promotion. The 43,576 reserved rows include financial advice and spoilers, whose labels remain outside research. [Pre-score protocol](../docs/EXPANDED_STUDY.md) · [Data provenance](../docs/RELEASED_DATA.md).

Use three familiar-policy grouped folds and four held-out-policy folds. Every training body and supplied example is checked against validation bodies. Retain repeated annotations in the primary experiment; handle their influence through grouped validation and declared sensitivity analyses.

In [ ]:
audit = expanded["audit"]
print("Protocol commit:", expanded["metadata"]["protocol_commit"])
print("Development rows / policies:", audit["development_rows"], audit["development_policies"])
print("Primary / total fitted models:", audit["primary_fitted_models"], audit["actual_fitted_models"])
print("Confirmation labels accessed during feature research:", audit["confirmation_labels_accessed"])
display(pd.DataFrame(audit["folds"])[["protocol", "fold", "training_rows", "validation_rows", "purged_training_rows"]])

## 2 · Search broadly, with a reason for each family
Words and character patterns capture phrasing and morphology. Style measures test requests, links and emphasis. Rule/support similarities, token products and semantic geometry test whether the comment resembles prohibited examples more than permitted ones. Training-reference ranks test relative position; community frequencies and cross-fitted target context test group effects and shortcut risk.

The encoder is already rule-conditioned. Its raw coordinates, support products and compact scalar comparisons are separate representations. Full-vocabulary, NB-weighted and five fixed SVD controls test whether screening or representation scale explains an apparent gain. [Detailed family rationale, availability and leakage analysis](../docs/FEATURE_RESEARCH.md#candidate-space-and-availability).

No timestamps, author histories, threads, opponents, coaches or ratings exist in the schema. Temporal and historical sports/customer-style features would invent unavailable information.

In [ ]:
screens = pd.DataFrame(expanded["screening"]["families"] + retrieval["screening"] + formatting["screening"])
assert (screens.candidates == screens.retained + screens.rejected).all()
totals = screens.groupby(["protocol", "fold"])[["candidates", "retained", "rejected"]].sum()
display(totals.rename(columns={"candidates": "All candidates", "retained": "All retained", "rejected": "All rejected"}))
print("Retained widths describe separate banks, not the selected centroid.")
display_expanded(root, "screening")

## 3 · Fit every learned transform inside training
Vocabulary/IDF, rarity and redundancy decisions, effect-score screening, scaling, ranks, NB weights and SVD use the purged outer-training rows. Target/context features use inner grouped cross-fitting and inner text purging. Unknown groups fall back to inner-training priors. No validation target selects a column.

A retained bank is not a final feature set: each model consumes only its declared families. The full candidate catalogs, selected names and training IDs remain checksummed private artifacts. Logistic regression stays at C=2; no hyperparameter search compensates for weak representations.

In [ ]:
decisions = pd.DataFrame([{"protocol": row["protocol"], "fold": row["fold"], "family": row["family"], **row["decisions"]} for row in expanded["screening"]["families"]]).fillna(0)
display(decisions.drop(columns=["protocol", "fold"]).groupby("family").sum().astype(int))

## 4 · Attribute improvement through matched additions and removals
Each addition starts from the same screened-word model. Each removal starts from the same all-transfer representation. These contrasts test incremental utility, while comparisons with the historical-style reference also change representation/scaling conventions. Paired normalized-body bootstrap draws share weights across models; simultaneous intervals cover the declared contrasts. They condition on fixed OOF predictions and four observed policies, not arbitrary future rules or all adaptive research decisions.

In [ ]:
display_expanded(root, "ablation")
contrasts = pd.DataFrame(expanded["uncertainty"])
removals = contrasts[(contrasts.protocol == "heldout_rule") & contrasts.contrast.str.startswith("remove_")]
display(removals[["contrast", "observed_delta", "ci_lower", "ci_upper", "simultaneous_lower", "simultaneous_upper"]].round(4))

## 5 · Check whether the selected signals are stable
Selection overlap measures whether folds retain the same columns; it does not establish usefulness. Group permutation shuffles each group's linear contribution within policy, preserving policy prevalence. Correlated families can substitute for one another, so the direct ablations remain the primary contribution evidence. Transparent linear contributions make a second SHAP plot unnecessary here.

In [ ]:
stability = pd.DataFrame(expanded["screening"]["stability"])
display(stability[stability.protocol == "heldout_rule"].groupby("family").retained_jaccard.agg(["min", "mean", "max"]).round(3))
importance = pd.DataFrame([{**r, "mean_permutation_auc_drop": sum(r["within_rule_permutation_auc_drops"]) / len(r["within_rule_permutation_auc_drops"])} for r in expanded["importance"] if r["protocol"] == "heldout_rule" and r["model"] == "all_transfer"])
display(importance.groupby("family")[["mean_absolute_logit_contribution", "mean_permutation_auc_drop"]].mean().round(4))

## 6 · Challenge duplicate, label and support dependence
Training-conflict exclusions use training labels only. Near-copy removal uses text only and fixed character-cosine/Jaccard thresholds. The validation rows stay fixed for both refits. Separately, descriptive metrics give equal total weight to each body/policy group or exclude conflicting groups from scoring. None of these diagnostic outcomes changes the primary folds.

The centroid stress tests average all four one-positive/one-negative example choices, shuffle supplied contexts within policy, and exclude exact self-support matches. They test dependence on the supplied examples; they do not establish paraphrase independence or resilience to absent rule text.

In [ ]:
filters = pd.DataFrame(audit["training_sensitivities"])
filters["removed"] = filters.before - filters.after
display(filters.groupby(["protocol", "sensitivity", "model"])[["removed", "reused_primary"]].sum())
robust_records = [r for r in expanded["results"] if r["model"] in ["rule_examples", "character_full", "qwen_centroid"] or "exclude_conflicts" in r["model"] or "purge_near_copies" in r["model"]]
display(metric_table(robust_records, heldout=True))
display(pd.DataFrame([{ "Model": r["model"], "Primary AUC": r["metrics"]["rule_macro_auc"], "Equal group weight": r["equal_body_policy_weight"]["rule_macro_auc"], "Exclude conflicting groups": r["excluding_conflicts"]["rule_macro_auc"]} for r in robust_records if r["protocol"] == "heldout_rule"]).round(4))

## 7 · Keep negative findings and scope limits visible
The original two-policy study also tested a frozen DeBERTa NLI cross-encoder and three fixed Qwen instruction-likelihood templates. Neither improved its lexical reference. These are useful negative findings for those models, inputs and policies; they do not establish failure on every policy. The expanded study tests the existing broad families and representation controls, rather than quietly attributing new-policy results to unexecuted NLI/instruction experiments.

In [ ]:
historical = [r for r in pairs["results"] if r["model"] in ["rule_nli", "word_nli", "all_nli"]]
if instructions is not None:
    historical += instructions["results"]
print("Historical two-policy evidence only:")
display(metric_table(historical, heldout=True))

## 8 · Test a target-derived geometry hypothesis
The raw-coordinate and linear projection controls leave one plausible avenue: local neighborhoods and prototypes of labeled training examples. The preregistered retrieval extension uses three frozen geometric spaces and 309 neighborhood, prototype, nonlinear and supplied-margin interaction candidates. Training-only screening retains at most 64. It adds the family to four exact saved controls and also tests retrieval alone: 35 additional fixed fits.

Every labeled bank excludes the query policy, including familiar policies. Training features leave out each entire policy and purge reference body/support matches; validation transformation rejects a target column. Tests flip all labels of a query policy and prove its own feature rows are unchanged. This tests transferable geometry without assigning labels from one policy to another. [Protocol and rationale](../docs/RETRIEVAL_STUDY.md).

In [ ]:
display(metric_table(retrieval["results"], heldout=True))
rc = pd.DataFrame(retrieval["uncertainty"])
display(rc[(rc.protocol == "heldout_rule") & rc.contrast.str.startswith("add_retrieval")][["contrast", "observed_delta", "ci_lower", "ci_upper", "simultaneous_lower", "simultaneous_upper"]].round(4))
display(pd.DataFrame(retrieval["screening"])[["protocol", "fold", "candidates", "retained", "rejected"]])

## 9 · Check the model's trained embedding resolutions
Matryoshka prefixes are different from learned SVD projections or arbitrary coordinate selection. Six preregistered resolutions (32–1,024 dimensions) reuse the same frozen vectors; smaller prefixes are normalized again. The 1,024-dimensional score must reproduce the existing centroid reference exactly. These are five additional scalar scoring controls, with zero new encoder calls or classifier fits—not thousands of new candidate columns. [Protocol and model-card rationale](../docs/RESOLUTION_STUDY.md).

In [ ]:
display(pd.DataFrame([{ "Dimensions": r["dimension"], "Policy-macro AUC": r["metrics"]["rule_macro_auc"], "Log loss": r["metrics"]["log_loss"], "Brier": r["metrics"]["brier"]} for r in resolution["results"]]).round(4))
display(pd.DataFrame(resolution["uncertainty"])[["candidate", "observed_delta", "ci_lower", "ci_upper", "simultaneous_lower", "simultaneous_upper"]].round(4))

## 10 · Test the remaining semantic hypotheses
The preregistered extension compares instructed comment queries with plain support documents, and generic rule entailment with affirmative policy behavior. It adds 32 asymmetric and 45 intent candidates, four fixed models across seven folds, and three frozen scores. The original banks and controls are reused unchanged.

Plain supports lower centroid transfer AUC to 0.6865. Affirmative wording improves a weak generic NLI score from 0.4341 to 0.5174, still far below the centroid. Adding intent to semantic scalars gives only +0.0033 AUC with an interval spanning zero, while probability losses worsen. [Protocol and results](../docs/SEMANTIC_FORMATTING.md).

In [ ]:
display_formatting(root, "contrasts")
display(pd.DataFrame(formatting["screening"])[["protocol", "fold", "family", "candidates", "retained"]])
stability = pd.DataFrame(formatting["audit"]["stability"])
display(stability.groupby("family").jaccard.agg(["min", "mean", "max"]).round(3))

## 11 · Explain the errors without relabeling them
Before seeing the new scores, a seeded 48-row legal/medical review examined speech act, policy behavior, context dependence and ambiguity. Every sampled body, policy and target matched its pinned source. Requests, personal experience, discussion and implicit advice can share vocabulary while differing in prohibited behavior.

This is a stratified qualitative review by one assistant, not independent human adjudication or a population label-error estimate. Twenty-three rows were flagged as ambiguous; none was relabeled. Raw text and row annotations remain private. The review motivated a falsifiable representation check, whose negative outcome remains visible.

In [ ]:
review = formatting["error_audit"]
display(pd.Series(review["taxonomy_counts"]["speech_act"], name="Reviewed speech acts"))
display(pd.Series(decision["verification"], name="Private artifact replay"))

## 12 · Apply the stopping rule, including complementarity
The replacement criteria were committed before the new semantic scores: at least +0.005 macro AUC, a positive simultaneous lower bound, no policy loss above 0.02 AUC, and log-loss/Brier increases no larger than 0.01/0.005. None of seven alternatives passes. One separately preregistered 50/50 average checks whether centroid and semantic-intent scores complement each other; no weights or policy routing are tuned.

The average reaches 0.7086 AUC, but its +0.0044 gain is uncertain, advertising loses 0.0222, and both probability losses worsen. It fails all five conditions. Retain the simpler centroid. The [coverage ledger](../docs/FEATURE_COVERAGE.md) documents applicable families, exclusions and deferred model research; this is bounded evidence of diminishing returns, not universal feature exhaustion.

Final model fitting, nested calibration and the protected comparison are now complete; notebook `03` presents the accepted result. Verified offline integration remains required. The current standalone inference notebook remains the explicitly named lexical reference.

In [ ]:
eligibility = decision["decision"]["candidate_eligibility"] + decision["fusion"]["decision"]["candidate_eligibility"]
display(pd.DataFrame([{ "Candidate": r["candidate"], "AUC": r["macro_auc"], "Gain": r["auc_gain"], "Eligible": r["eligible"], "Failed checks": ", ".join(r["failed_conditions"])} for r in eligibility]).round(4))

In [ ]:
display(pd.DataFrame(gate["criteria"]))
print("Final training justified:", gate["final_training_authorized_by_evidence"])
print(gate["decision"])

## Reproduce the evidence
`python scripts/run_expanded.py --encode` extends only missing pinned embeddings, then resumes the preregistered study after the private research artifacts are restored. Omit `--encode` to require a complete verified cache. The default public notebook performs no fitting, downloads or target access. Sources, input identities, stage markers, full catalogs and OOF predictions are preserved. [Execution and restoration record](../docs/VALIDATION.md).